# TELEPATI 8.0: AgriData Intelligence Race
## Deteksi Penyakit Tanaman Padi Berbasis *Object Detection*

**Pernyataan kepatuhan.** Model dibangun sepenuhnya dari definisi arsitektur
`.yaml` dengan `pretrained=False`, tanpa *external pretrained weights* dalam
bentuk apa pun. Tidak ada dataset di luar dataset resmi kompetisi. Tidak ada
pemrosesan dataset menggunakan LLM maupun API eksternal. Pemetaan 11 kelas
canonical diverifikasi terhadap dataset aktual tanpa kategori tersisa yang
tidak terpetakan. *Seeding* bersifat deterministik dan `YOLO_OFFLINE=1`
diaktifkan pada seluruh skrip pelatihan dan evaluasi.

**Cara membaca notebook ini.** Notebook disusun sebagai alur penelitian,
bukan sebagai kumpulan sel kode. Urutannya mengikuti rantai berikut:
masalah, data, temuan data, implikasi, keputusan metodologi, eksperimen,
hasil, analisis kesalahan, keterbatasan, lalu kesimpulan. Logika analisis
disimpan pada paket `src/agridata/` dan skrip pada `scripts/`, sehingga
notebook memanggil fungsi yang sudah diuji, bukan menyalin ulang kode.

**Mode eksekusi.** Secara *default* notebook berjalan pada mode evaluasi
(`SKIP_TRAINING = True`), yaitu memuat *final weights* yang sudah dilatih
sehingga dapat dijalankan dari atas ke bawah dalam hitungan menit. Mode
reproduksi penuh tersedia dan terdokumentasi pada Bagian 13.

## 1. Latar Belakang dan Tujuan

Padi merupakan komoditas pangan utama, dan penyakit pada daun serta malai
padi dapat menurunkan hasil panen secara signifikan. Identifikasi penyakit
di lapangan umumnya dilakukan secara manual melalui pengamatan visual oleh
petani atau penyuluh. Pendekatan tersebut membutuhkan pengalaman, memakan
waktu, dan sulit diskalakan untuk area tanam yang luas.

Pendekatan berbasis *computer vision* menawarkan alternatif yang dapat
diskalakan, karena satu model dapat memeriksa banyak citra dalam waktu
singkat. Berbeda dengan klasifikasi citra yang hanya memberikan satu label
per gambar, *object detection* memberikan dua informasi sekaligus, yaitu
jenis penyakit dan lokasi gejalanya pada citra. Informasi lokasi ini
relevan untuk kasus nyata, karena satu helai daun dapat memuat beberapa
bercak penyakit, dan satu citra lapangan dapat memuat lebih dari satu
kondisi.

**Tujuan pekerjaan ini** adalah membangun model *object detection* untuk 11
kelas canonical penyakit dan kondisi tanaman padi menggunakan dataset resmi
TELEPATI 8.0, dengan pipeline yang dapat direproduksi dan diaudit ulang oleh
pihak ketiga.

## 2. Rumusan Masalah

Pekerjaan ini diarahkan untuk menjawab pertanyaan berikut:

1. Bagaimana kondisi aktual dataset resmi yang tersedia, termasuk
   distribusi kelas, karakteristik geometri objek, dan kualitas anotasinya?
2. Masalah kualitas data apa yang perlu ditangani sebelum pemodelan, dan
   apa konsekuensinya bila diabaikan?
3. Konfigurasi pelatihan seperti apa yang dapat dipertanggungjawabkan
   berdasarkan eksperimen terkontrol, bukan berdasarkan asumsi?
4. Seberapa baik performa model yang dihasilkan, diukur dengan mAP@0.5 dan
   *F1-score*, dan bagaimana performa tersebut terdistribusi antar kelas?
5. Pada kondisi seperti apa model gagal, dan faktor apa yang konsisten
   dengan kegagalan tersebut?
6. Seberapa jauh seluruh pipeline dapat direproduksi dan diaudit?

**Batasan.** Kompetisi melarang penggunaan *external pretrained weights*,
sehingga model harus dilatih dari inisialisasi acak. Pembatasan ini
berdampak langsung pada performa yang dapat dicapai, karena model tidak
mewarisi representasi visual umum dari dataset besar seperti COCO atau
ImageNet. Seluruh interpretasi hasil pada notebook ini harus dibaca dalam
konteks batasan tersebut.

## 3. Gambaran Solusi

Solusi disusun sebagai pipeline bertahap yang setiap langkahnya
menghasilkan artefak yang dapat diperiksa kembali:

```
Dataset mentah
  -> Audit forensik dataset
  -> Pemetaan 11 kelas canonical
  -> Pemeriksaan kebocoran antar split
  -> Penyiapan data format YOLO
  -> Eksperimen terkontrol (21 percobaan)
  -> Pelatihan model final
  -> Evaluasi dan analisis kesalahan
  -> Audit reproducibility
```

Arsitektur yang digunakan adalah YOLOv8n, yaitu varian terkecil pada
keluarga YOLOv8. Pemilihan varian nano mempertimbangkan dua hal: model
dilatih dari nol tanpa bobot awal, dan perangkat yang tersedia adalah
laptop Apple Silicon dengan *backend* MPS. Model berkapasitas besar yang
dilatih dari nol pada dataset berukuran sedang justru berisiko lebih sulit
dioptimasi dalam anggaran komputasi yang tersedia.

Setiap tahap pada diagram di atas memiliki skrip tersendiri di `scripts/`
dan laporan terstruktur di `artifacts/`, sehingga klaim pada notebook ini
dapat ditelusuri sampai ke berkas hasil eksekusi.

## 4. Lingkungan Pengembangan dan Reproducibility

Bagian ini mencatat identitas lingkungan eksekusi. Informasi ini merupakan
bagian dari jejak audit: metrik apa pun yang dilaporkan pada notebook ini
hanya bermakna bila diketahui pada kondisi perangkat dan versi pustaka
seperti apa metrik tersebut dihasilkan.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "agridata").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src" / "agridata").exists(), (
    "Paket agridata tidak ditemukan. Jalankan notebook dari root project atau dari notebooks/."
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Root project: {PROJECT_ROOT}")

In [ ]:
import json
import subprocess
import hashlib
import random
import csv

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import yaml

from agridata.seed import set_global_seed
from agridata.device import detect_device
from agridata.reproducibility.environment import capture_environment_snapshot
from agridata.dataset.mapping import CANONICAL_CLASSES, build_mapping_report
from agridata.dataset.stats import load_canonical_split
from agridata.analysis.dataset_profile import (
    audit_missingness,
    compute_bbox_geometry,
    load_duplicate_summary,
    scene_density_summary,
    summarize_class_imbalance,
    summarize_resolution,
)
from agridata.visualization.images import draw_annotated_image
from agridata.visualization.distributions import (
    plot_class_distribution_comparison,
    plot_instances_vs_performance,
    plot_per_class_ap,
    plot_split_overview,
    plot_training_curves,
)
from agridata.training.train import build_compliant_model, run_training

In [ ]:
env = capture_environment_snapshot()
print(f"Versi Python        : {env['python_version']}")
print(f"Platform            : {env['platform']['platform_string']}")
print(f"Perangkat terpilih  : {env['device']['resolved_device']} (Apple Silicon: {env['device']['is_apple_silicon']})")
print(f"torch               : {env['device']['torch_version']}")
print(f"CUDA tersedia       : {env['device']['cuda_available']}  |  MPS tersedia: {env['device']['mps_available']}")
print(f"Git commit          : {env['git_commit']}")
print(f"Working tree bersih : {env['git_status']['clean']}")

Seluruh sumber keacakan dikunci pada satu nilai *seed* yang sama dengan
yang digunakan pada pelatihan final. Ini mencakup modul `random` pada
Python, NumPy, PyTorch, serta variabel `PYTHONHASHSEED`.

In [ ]:
with open(PROJECT_ROOT / "configs" / "final_model_config.yaml") as f:
    FINAL_CONFIG = yaml.safe_load(f)

SEED = FINAL_CONFIG["seed"]
set_global_seed(SEED)

DATASET_ROOT = PROJECT_ROOT / "Telepati 8.0 Datasets"
PREPARED_DIR = PROJECT_ROOT / "data" / "prepared"
REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"
FIGURES_DIR = PROJECT_ROOT / "artifacts" / "figures" / "final_submission"

assert DATASET_ROOT.exists(), f"Dataset resmi tidak ditemukan pada {DATASET_ROOT}"
print(f"Seed global    : {SEED}")
print(f"Root dataset   : {DATASET_ROOT}")

## 5. Dataset dan Sumber Data

Dataset yang digunakan adalah dataset resmi TELEPATI 8.0. Tidak ada sumber
data lain yang ditambahkan. Dataset sudah terbagi menjadi tiga *split*
resmi, yaitu `train`, `valid`, dan `test`, dan pembagian tersebut
dipertahankan apa adanya. Penggabungan atau pengacakan ulang antar *split*
tidak dilakukan, karena akan merusak dasar perbandingan dan berpotensi
menimbulkan kebocoran informasi.

Anotasi disimpan dalam format COCO, satu berkas `_annotations.coco.json`
per *split*, yang memuat daftar citra, daftar anotasi *bounding box*, dan
daftar kategori.

In [ ]:
raw_counts = {}
for split in ["train", "valid", "test"]:
    with open(DATASET_ROOT / split / "_annotations.coco.json") as f:
        data = json.load(f)
    raw_counts[split] = {
        "citra": len(data["images"]),
        "anotasi": len(data["annotations"]),
        "kategori_mentah": len(data["categories"]),
    }

print(f"{'Split':8s} {'Citra':>8s} {'Anotasi':>10s} {'Kategori mentah':>18s}")
for split, c in raw_counts.items():
    print(f"{split:8s} {c['citra']:8d} {c['anotasi']:10d} {c['kategori_mentah']:18d}")
print(f"\nTotal citra   : {sum(c['citra'] for c in raw_counts.values())}")
print(f"Total anotasi : {sum(c['anotasi'] for c in raw_counts.values())}")

Dataset memuat 21 kategori mentah, sedangkan target deteksi resmi berjumlah
11 kelas canonical. Selisih ini bukan kesalahan dataset, melainkan
konsekuensi dari variasi penulisan label dan keberadaan kategori
*supercategory* yang bukan target deteksi. Penanganannya dibahas pada
Bagian 7.

## 6. Eksplorasi dan Profiling Dataset

Bagian ini memeriksa kondisi aktual dataset sebelum keputusan pemodelan
diambil. Tujuannya bukan sekadar menampilkan grafik, melainkan
mengidentifikasi karakteristik data yang nantinya diperlukan untuk
menafsirkan performa model secara jujur.

Seluruh perhitungan pada bagian ini memanggil fungsi pada
`src/agridata/analysis/dataset_profile.py` dan dapat dihasilkan ulang
melalui `python scripts/profile_dataset.py`.

### 6.1 Struktur Dataset

Data dimuat dengan pemetaan canonical sudah diterapkan, sehingga jumlah
anotasi yang ditampilkan adalah jumlah anotasi yang benar-benar menjadi
target deteksi.

In [ ]:
splits = {}
for split in ["train", "valid", "test"]:
    splits[split] = load_canonical_split(DATASET_ROOT, split, "_annotations.coco.json")

split_counts = {
    s: {"images": len(d.images), "annotations": len(d.annotations)} for s, d in splits.items()
}

print(f"{'Split':8s} {'Citra':>8s} {'Anotasi canonical':>20s} {'Rata-rata anotasi/citra':>26s}")
for s, c in split_counts.items():
    rata = c["annotations"] / c["images"]
    print(f"{s:8s} {c['images']:8d} {c['annotations']:20d} {rata:26.2f}")

### 6.2 Distribusi Split

Proporsi antar *split* menentukan seberapa kuat kesimpulan yang dapat
ditarik dari evaluasi. *Split* validasi yang terlalu kecil membuat metrik
menjadi tidak stabil, sedangkan *split* latih yang terlalu kecil membatasi
kemampuan model belajar.

In [ ]:
fig_path = plot_split_overview(split_counts, FIGURES_DIR / "split_overview.png")
plt.figure(figsize=(9, 5))
plt.imshow(mpimg.imread(fig_path))
plt.axis("off")
plt.show()

total_img = sum(c["images"] for c in split_counts.values())
for s, c in split_counts.items():
    print(f"{s:8s}: {c['images']/total_img:6.1%} dari total citra")

Proporsi pembagian mendekati pola 76 persen latih, 16 persen validasi, dan
8 persen uji. Ukuran *split* validasi sebanyak lebih dari dua ribu citra
cukup memadai untuk menghasilkan estimasi metrik yang stabil, dan hal ini
kemudian terbukti pada Bagian 14, ketika mAP@0.5 tercatat identik pada
beberapa kali pengulangan evaluasi.

### 6.3 Distribusi Kelas

Dua besaran berbeda perlu dibedakan secara eksplisit. Jumlah *instance*
adalah banyaknya *bounding box* untuk suatu kelas, sedangkan jumlah citra
adalah banyaknya gambar yang memuat minimal satu *instance* kelas
tersebut. Keduanya tidak identik, karena satu citra dapat memuat banyak
*instance* dari kelas yang sama.

In [ ]:
imbalance = {s: summarize_class_imbalance(d) for s, d in splits.items()}
train_imb = imbalance["train"]

fig_path = plot_class_distribution_comparison(
    train_imb.per_class_instances,
    train_imb.per_class_images,
    "Distribusi kelas pada split train",
    FIGURES_DIR / "train_class_distribution.png",
)
plt.figure(figsize=(10, 6))
plt.imshow(mpimg.imread(fig_path))
plt.axis("off")
plt.show()

In [ ]:
print(f"{'Kelas':28s} {'Instance':>9s} {'Citra':>7s} {'Instance/citra':>15s} {'Porsi':>8s}")
for cls in CANONICAL_CLASSES:
    n_inst = train_imb.per_class_instances[cls]
    n_img = train_imb.per_class_images[cls]
    rasio = n_inst / n_img if n_img else 0
    print(f"{cls:28s} {n_inst:9d} {n_img:7d} {rasio:15.2f} {train_imb.per_class_instance_share[cls]:7.1%}")

print(f"\nKelas terbanyak : {train_imb.most_frequent_class} ({train_imb.max_instances} instance)")
print(f"Kelas tersedikit: {train_imb.least_frequent_class} ({train_imb.min_instances} instance)")
print(f"Rasio ketidakseimbangan: {train_imb.imbalance_ratio:.1f} kali")

### 6.4 Analisis Ketidakseimbangan Kelas

Rasio antara kelas terbanyak dan kelas tersedikit pada *split* latih
mencapai 22,6 kali. Rasio ini juga berbeda antar *split*, sehingga perlu
diperiksa secara terpisah.

In [ ]:
print(f"{'Split':8s} {'Terbanyak':<26s} {'Tersedikit':<26s} {'Rasio':>8s}")
for s, imb in imbalance.items():
    print(
        f"{s:8s} {imb.most_frequent_class + ' (' + str(imb.max_instances) + ')':<26s} "
        f"{imb.least_frequent_class + ' (' + str(imb.min_instances) + ')':<26s} {imb.imbalance_ratio:7.1f}x"
    )

**Implikasi untuk pemodelan.** Model menerima frekuensi observasi yang
sangat berbeda antar kelas. Kelas dengan jumlah *instance* rendah memperoleh
lebih sedikit variasi visual selama pelatihan, sehingga kemampuan
generalisasinya berpotensi lebih rendah. Kondisi ini menjadi konteks penting
ketika membaca perbedaan AP@0.5 antar kelas pada Bagian 15. Perlu dicatat
bahwa jumlah data bukan satu-satunya faktor, dan hubungan antara keduanya
dianalisis secara eksplisit pada Bagian 17.

Perlu dicatat pula bahwa rasio ketidakseimbangan pada *split* validasi dan
uji lebih tinggi daripada pada *split* latih, yaitu sekitar 32 kali dan 31
kali. Artinya evaluasi dilakukan pada distribusi yang bahkan lebih timpang
daripada distribusi pelatihan.

### 6.5 Distribusi Ukuran *Bounding Box*

Ukuran objek merupakan salah satu faktor paling menentukan dalam
*object detection*. Agar dapat dibandingkan antar citra dengan resolusi
berbeda, luas *bounding box* dihitung relatif terhadap luas citranya
sendiri. Ambang objek kecil ditetapkan pada satu persen luas citra.

In [ ]:
geometry = {s: compute_bbox_geometry(d, small_object_threshold=0.01) for s, d in splits.items()}

plt.figure(figsize=(10, 5))
plt.imshow(mpimg.imread(FIGURES_DIR / "train_bbox_relative_area.png"))
plt.axis("off")
plt.show()

print(f"{'Split':8s} {'Median luas relatif':>22s} {'Median rasio aspek':>21s} {'Porsi objek kecil':>20s}")
for s, g in geometry.items():
    print(f"{s:8s} {g.median_relative_area:22.4%} {g.median_aspect_ratio:21.2f} {g.small_object_share:20.1%}")

Sebaran luas relatif sangat condong ke kiri. Pada *split* latih, 38,1 persen
*bounding box* menutupi kurang dari satu persen luas citra, dan proporsinya
justru lebih tinggi pada *split* validasi (43,3 persen) serta uji (46,8
persen).

**Implikasi untuk pemodelan.** Dominasi objek kecil membuat lokalisasi
menjadi sensitif terhadap resolusi masukan dan terhadap pergeseran beberapa
piksel saja. Kondisi ini menjadi salah satu alasan mengapa ukuran citra
masukan diperlakukan sebagai parameter yang diuji secara eksperimental pada
Bagian 11, bukan ditetapkan berdasarkan asumsi.

In [ ]:
plt.figure(figsize=(10, 5))
plt.imshow(mpimg.imread(FIGURES_DIR / "train_bbox_aspect_ratio.png"))
plt.axis("off")
plt.show()

Distribusi rasio aspek terpusat di sekitar nilai satu, yang berarti sebagian
besar *bounding box* mendekati bentuk persegi. Namun terdapat ekor ke arah
kanan, yaitu kotak yang jauh lebih lebar daripada tingginya. Bentuk
memanjang seperti ini konsisten dengan gejala penyakit yang menyebar
mengikuti bentuk helai daun.

### 6.6 Resolusi Citra

Variasi resolusi memengaruhi bagaimana citra diubah ukurannya sebelum masuk
ke model, dan karenanya memengaruhi ukuran efektif objek kecil.

In [ ]:
resolution = {s: summarize_resolution(d) for s, d in splits.items()}

plt.figure(figsize=(7, 7))
plt.imshow(mpimg.imread(FIGURES_DIR / "train_image_resolution.png"))
plt.axis("off")
plt.show()

print(f"{'Split':8s} {'Resolusi unik':>15s} {'Resolusi dominan':>20s} {'Porsi dominan':>16s}")
for s, r in resolution.items():
    dom = f"{r.most_common_resolution[0]}x{r.most_common_resolution[1]}"
    print(f"{s:8s} {r.distinct_resolutions:15d} {dom:>20s} {r.most_common_share:15.1%}")

### 6.7 *Missingness* dan Validitas Anotasi

Istilah *missing value* pada data tabular tidak dapat dipindahkan begitu
saja ke dataset deteksi objek. Pada konteks ini, *missingness* diartikan
sebagai relasi yang putus antara berkas JSON dan berkas citra, atau
parameter *bounding box* yang tidak dapat mendeskripsikan suatu wilayah.

Setiap pemeriksaan dilaporkan meskipun hasilnya nol, karena nilai nol
merupakan hasil audit yang sah dan bukan ketiadaan pemeriksaan.

In [ ]:
missing = {s: audit_missingness(DATASET_ROOT, s, "_annotations.coco.json") for s in splits}

for s, report in missing.items():
    print(f"--- split {s} ---")
    for c in report.checks:
        status = "OK" if c.count == 0 else "PERLU DITINJAU"
        print(f"  {c.count:6d} / {c.total:6d} ({c.percentage:5.2f}%)  {c.name:58s} {status}")
    print()

Hasil audit menunjukkan integritas referensi yang bersih: tidak ada berkas
citra yang hilang, tidak ada berkas tanpa *record* JSON, tidak ada anotasi
yang merujuk citra atau kategori yang tidak ada, tidak ada *bounding box*
kosong maupun berdimensi tidak valid, dan tidak ada metadata dimensi citra
yang hilang.

Dua pemeriksaan menghasilkan nilai bukan nol dan perlu dijelaskan:

1. **Citra tanpa anotasi**, yaitu 59 citra pada *train*, 13 pada *valid*,
   dan 5 pada *test*. Citra semacam ini tidak memuat objek target. Pada
   kerangka *object detection*, citra tanpa anotasi tetap dapat berfungsi
   sebagai contoh latar belakang, sehingga keberadaannya tidak otomatis
   merupakan cacat data. Yang perlu dicatat adalah jumlah citra efektif yang
   memuat target deteksi sedikit lebih rendah daripada jumlah citra total.
2. **Kategori tanpa anotasi**, yaitu tiga kategori pada setiap *split*.
   Ketiganya adalah `Leaf-blight`, `Rice-Leaf-Diseasee`, dan `paddy`, yang
   merupakan *supercategory* dan bukan target deteksi. Temuan ini konsisten
   dengan keputusan pemetaan pada Bagian 7.

### 6.8 *Duplicate* dan *Data Leakage*

Kesamaan citra antar *split* merupakan risiko serius, karena model dapat
dievaluasi pada citra yang sudah pernah dilihatnya saat pelatihan. Audit
forensik pada tahap awal project memeriksa hal ini menggunakan *hash* konten
berkas.

In [ ]:
duplicates = load_duplicate_summary(PROJECT_ROOT / "artifacts" / "audit" / "dataset_audit_report.json")

if duplicates["available"]:
    for pair, n in duplicates["pairs"].items():
        print(f"{pair:18s}: {n} citra duplikat persis")
    print(f"\nTotal duplikat persis lintas split: {duplicates['total_exact_duplicates']}")
    for pair, matches in duplicates["matches"].items():
        for m in matches:
            print(f"\n  pasangan pada {pair}:")
            for k, v in m.items():
                print(f"    {k}: {v}")
else:
    print(duplicates["reason"])

Ditemukan satu pasangan citra dengan konten identik antara *split* latih dan
*split* uji. Penanganan yang dilakukan bersifat spesifik dan terdokumentasi:

- dataset mentah tidak diubah sama sekali;
- pada *manifest* data siap latih, entri pada sisi `train` dikeluarkan;
- *split* validasi dan uji tidak dimodifikasi.

Dengan cara ini, risiko kebocoran informasi berkurang tanpa mengubah dasar
evaluasi resmi. Pemeriksaan tambahan berbasis *perceptual hash* menemukan
sejumlah kandidat kemiripan yang tidak diverifikasi satu per satu secara
visual, dan keterbatasan ini dicatat pada Bagian 18.

### 6.9 Contoh Visual Dataset

Sampel diambil secara deterministik menggunakan *seed* global, sehingga
citra yang sama akan muncul pada setiap eksekusi.

In [ ]:
train_data = splits["train"]
ann_by_image = {}
for ann in train_data.annotations:
    ann_by_image.setdefault(ann.image_id, []).append(ann)

set_global_seed(SEED)
sample_ids = random.sample(sorted(ann_by_image.keys()), 4)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, image_id in zip(axes, sample_ids):
    rec = train_data.images_by_id[image_id]
    annotated = draw_annotated_image(DATASET_ROOT / "train" / rec.file_name, rec, ann_by_image[image_id])
    ax.imshow(annotated)
    ax.set_title(f"{len(ann_by_image[image_id])} anotasi", fontsize=10)
    ax.axis("off")
plt.suptitle("Contoh citra latih beserta anotasi ground truth", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
density = {s: scene_density_summary(d) for s, d in splits.items()}
print(f"{'Split':8s} {'Median anotasi/citra':>22s} {'Maksimum':>10s} {'Citra padat (>3)':>18s}")
for s, dd in density.items():
    print(f"{s:8s} {dd['median_annotations_per_image']:22.0f} {dd['max_annotations_in_one_image']:10d} {dd['crowded_share']:17.1%}")

### Key Takeaways Bagian 6

- Dataset menyediakan 13.298 citra dengan 27.721 anotasi canonical, jumlah
  yang memadai untuk melatih model deteksi berukuran kecil.
- Distribusi kelas sangat timpang, dengan rasio 22,6 kali pada *split* latih
  dan lebih dari 30 kali pada *split* validasi serta uji.
- Objek berukuran kecil mendominasi, yaitu 38,1 persen pada *split* latih dan
  46,8 persen pada *split* uji.
- Integritas referensi anotasi bersih pada seluruh pemeriksaan, dengan dua
  catatan yang sudah dijelaskan yaitu citra tanpa anotasi dan *supercategory*
  tanpa anotasi.
- Satu duplikat persis lintas *split* ditemukan dan ditangani pada tingkat
  *manifest*, tanpa mengubah dataset mentah.

Karakteristik di atas menjadi dasar untuk membaca hasil model pada bagian
berikutnya. Sebelum sampai ke pemodelan, label yang tidak konsisten perlu
disatukan terlebih dahulu.

## 7. Pemetaan 11 Kelas Canonical

Dataset mentah memuat 21 kategori, sedangkan target deteksi resmi berjumlah
11 kelas. Selisih tersebut berasal dari tiga sumber: variasi penulisan nama
untuk konsep yang sama, variasi kapitalisasi dan tanda hubung, serta
kategori *supercategory* yang bukan target deteksi.

Tanpa penyatuan ini, label yang secara semantik sama akan diperlakukan
sebagai kelas yang berbeda, sehingga data untuk satu penyakit terpecah dan
model dipaksa memisahkan sesuatu yang sebenarnya identik.

In [ ]:
with open(DATASET_ROOT / "train" / "_annotations.coco.json") as f:
    train_raw = json.load(f)

mapping_report = build_mapping_report(train_raw["categories"])

print(f"Kategori mentah          : {mapping_report['total_raw_categories']}")
print(f"Terpetakan ke canonical  : {len(mapping_report['mapped'])}")
print(f"Supercategory dikecualikan: {[p['raw_name'] for p in mapping_report['supercategory_placeholders']]}")
print(f"Tidak terpetakan         : {mapping_report['unmapped_raw_categories']} (harus kosong)")
assert not mapping_report["unmapped_raw_categories"], "Ada kategori mentah yang belum terpetakan."

print(f"\n{'Label mentah':32s} -> {'Kelas canonical':28s} {'ID model'}")
for m in sorted(mapping_report["mapped"], key=lambda r: r["canonical_id"]):
    print(f"{m['raw_name']:32s} -> {m['canonical_name']:28s} {m['canonical_id'] - 1}")

Pemetaan bersifat eksplisit dan gagal secara keras bila ditemukan kategori
mentah yang tidak dikenali, sehingga perubahan dataset di masa depan tidak
akan lolos diam-diam. Versi tabel pemetaan dicatat sebagai `MAPPING_VERSION`
agar setiap perubahan dapat dilacak.

Tiga kategori yang dikecualikan bukan dihapus secara sewenang-wenang.
Ketiganya terbukti tidak memiliki satu pun anotasi pada seluruh *split*,
sebagaimana ditunjukkan pada audit *missingness* di Bagian 6.7.

## 8. Persiapan Data

Tahap ini mengubah dataset mentah menjadi data siap latih dalam format YOLO,
dengan urutan berikut:

```
Dataset mentah
  -> Validasi struktur dan anotasi
  -> Pemetaan canonical
  -> Pemeriksaan kebocoran antar split
  -> Penulisan manifest dan label format YOLO
```

Prinsip yang dipegang pada tahap ini:

- dataset mentah tidak pernah diubah, dipindah, atau ditimpa;
- citra tidak disalin, melainkan dirujuk melalui *symlink*, sehingga tidak
  ada duplikasi byte;
- hasil penyiapan dapat dihasilkan ulang dari *seed* yang sama dan
  menghasilkan *manifest* yang identik byte per byte;
- pengecualian akibat duplikat hanya diterapkan pada sisi `train`.

In [ ]:
if not (PREPARED_DIR / "data.yaml").exists():
    print("Menyiapkan dataset siap latih...")
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "prepare_dataset.py"),
         "--dataset-root", str(DATASET_ROOT), "--output-dir", str(PREPARED_DIR), "--seed", str(SEED)],
        check=True, cwd=PROJECT_ROOT,
    )
else:
    print(f"Data siap latih sudah tersedia pada {PREPARED_DIR}")

with open(REPORTS_DIR / "dataset_preparation_summary.json") as f:
    prep = json.load(f)

print(f"\nSeed             : {prep['seed']}")
print(f"Versi pemetaan   : {prep['mapping_version']}")
print(f"\n{'Split':8s} {'Citra disiapkan':>17s} {'Dikecualikan (leakage)':>24s} {'Anotasi':>10s}")
for s in prep["splits"]:
    print(f"{s['split']:8s} {s['num_images_prepared']:17d} {s['num_images_excluded_leakage']:24d} {s['num_annotations_prepared']:10d}")

### Key Takeaways Bagian 7 dan 8

- 21 kategori mentah disatukan menjadi 11 kelas canonical, tanpa kategori
  tersisa yang tidak terpetakan.
- Tiga *supercategory* dikecualikan berdasarkan bukti bahwa keduanya tidak
  memuat anotasi sama sekali.
- Satu citra dikeluarkan dari *manifest* latih akibat duplikat lintas
  *split*, sementara dataset mentah tetap utuh.
- Proses penyiapan bersifat deterministik dan dapat diverifikasi ulang.

## 9. Strategi Eksperimen

Konfigurasi pelatihan tidak ditetapkan berdasarkan nilai *default* maupun
intuisi. Strategi yang digunakan adalah pengujian satu faktor pada satu
waktu, yaitu mengubah satu parameter sambil menahan parameter lain tetap,
sehingga perubahan hasil dapat diatribusikan pada faktor yang diubah.

Pengujian dilakukan pada skala penyaringan, yaitu menggunakan sebagian data
latih dan jumlah *epoch* yang kecil. Pilihan ini merupakan konsekuensi dari
anggaran komputasi yang tersedia. Konsekuensinya dicatat sebagai
keterbatasan pada Bagian 18: hasil pada skala penyaringan tidak dijamin
berlaku sama pada skala penuh.

Pertanyaan eksperimen yang diajukan:

1. Apakah ukuran citra masukan memengaruhi performa deteksi?
2. Apakah jumlah *epoch* masih memberikan perbaikan pada rentang yang diuji?
3. Apakah *optimizer* tertentu lebih stabil daripada yang lain?
4. Apakah *learning rate* yang lebih kecil membantu?
5. Apakah *augmentation* memberikan manfaat pada skala data ini?
6. Apakah penyeimbangan kelas melalui *oversampling* memperbaiki kelas minoritas?

## 10. Baseline dan Eksperimen Terkontrol

Seluruh percobaan dicatat pada `artifacts/experiments/experiment_log.json`
beserta *seed*, *hyperparameter*, arsitektur, *hash manifest* dataset, dan
*commit* Git pada saat percobaan dijalankan.

In [ ]:
with open(PROJECT_ROOT / "artifacts" / "experiments" / "experiment_log.json") as f:
    experiments = json.load(f)

print(f"Jumlah percobaan tercatat: {len(experiments)}\n")
print(f"{'ID':5s} {'imgsz':>6s} {'batch':>6s} {'epoch':>6s} {'optim':>7s} {'lr':>9s} {'mAP@0.5':>9s} {'Catatan'}")
for e in experiments:
    print(
        f"{e['experiment_id']:5s} {e['image_size']:6d} {e['batch_size']:6d} {e['epochs']:6d} "
        f"{e['optimizer']:>7s} {e['learning_rate']:9.6f} {e['best_val_map50']:9.4f} {e['notes'][:46]}"
    )

Analisis rinci mengenai temuan tiap kelompok percobaan dan alasan pemilihan
konfigurasi final diuraikan pada Bagian 12.

## 11. Konfigurasi Final

Konfigurasi final dibekukan pada `configs/final_model_config.yaml`. Setiap
parameter disertai alasan yang merujuk percobaan tertentu, sehingga dapat
ditelusuri kembali.

In [ ]:
for k, v in FINAL_CONFIG.items():
    print(f"{k:18s}: {v}")

## 12. Pemilihan Konfigurasi Final

Ringkasan alasan berbasis bukti untuk parameter utama:

- **Ukuran citra 640.** Percobaan dengan ukuran 640 mengungguli ukuran 320
  pada skala penyaringan, dan temuan ini diperkuat oleh analisis kesalahan
  yang menunjukkan bahwa objek yang terlewat cenderung lebih kecil daripada
  rata-rata.
- **Batch 16.** Percobaan dengan batch 32 justru menurunkan mAP@0.5 pada
  jumlah *epoch* yang sama, konsisten dengan berkurangnya jumlah pembaruan
  gradien per *epoch*.
- **AdamW dengan *learning rate* 0,001.** Alternatif SGD dan *learning rate*
  0,0001 keduanya menghasilkan mAP@0.5 lebih rendah pada skala penyaringan.
- **Augmentasi *default* dengan pengecualian *vertical flip*.** Pembalikan
  vertikal dinonaktifkan karena tidak masuk akal secara fisik untuk tanaman
  yang tumbuh mengikuti arah gravitasi. Keputusan ini mengutamakan penalaran
  domain di atas selisih metrik yang kecil pada skala penyaringan.
- **50 *epoch*.** Konfigurasi awalnya ditetapkan 20 *epoch* karena
  pertimbangan tenggat. Setelah hasil 20 *epoch* diperoleh, pelatihan
  diperpanjang menjadi 50 *epoch*, dan perpanjangan tersebut terbukti
  menaikkan mAP@0.5 dari 0,5620 menjadi 0,6277. Hasil 20 *epoch* tetap
  diarsipkan pada `artifacts/archive/20epoch_run/` sebagai jejak audit.

## 13. Pelatihan Model Final

Model dibangun dari definisi arsitektur `yolov8n.yaml` dengan
`pretrained=False`. Fungsi `build_compliant_model` menolak berjalan bila
diberi `pretrained=True` atau bila argumen arsitektur menyerupai berkas
*checkpoint*, sehingga pelanggaran aturan kompetisi gagal secara keras dan
bukan lolos diam-diam.

In [ ]:
compliant_model = build_compliant_model(FINAL_CONFIG["model_arch"], pretrained=False)
print(f"Model dibangun dari definisi arsitektur '{FINAL_CONFIG['model_arch']}'. Task: {compliant_model.task}")
print("Tidak ada checkpoint eksternal yang dirujuk maupun diunduh (YOLO_OFFLINE aktif).")

### Mode eksekusi

Notebook menyediakan dua mode:

- **Mode evaluasi** (`SKIP_TRAINING = True`, *default*): memuat *final
  weights* yang sudah dilatih, sehingga notebook dapat dibaca dan
  diverifikasi tanpa menunggu proses pelatihan berjam-jam.
- **Mode reproduksi penuh** (`SKIP_TRAINING = False`): menjalankan ulang
  pelatihan dari konfigurasi beku, memerlukan sekitar 7,3 jam pada perangkat
  Apple Silicon dengan *backend* MPS.

Kode pelatihan pada mode kedua bukan tiruan, melainkan fungsi yang sama
dengan yang menghasilkan *weights* final.

In [ ]:
SKIP_TRAINING = True

FINAL_WEIGHTS_PATH = PROJECT_ROOT / "runs" / "detect" / "final" / "final_model" / "weights" / "best.pt"

if SKIP_TRAINING:
    assert FINAL_WEIGHTS_PATH.exists(), (
        f"SKIP_TRAINING=True tetapi weights final tidak ditemukan pada {FINAL_WEIGHTS_PATH}. "
        "Setel SKIP_TRAINING=False untuk melatih dari awal, atau jalankan scripts/run_final_training.py."
    )
    print(f"Mode evaluasi: memuat weights final dari {FINAL_WEIGHTS_PATH}")
else:
    print("Mode reproduksi penuh: menjalankan pelatihan dari konfigurasi beku.")
    extra_kwargs = {
        "optimizer": FINAL_CONFIG["optimizer"], "lr0": FINAL_CONFIG["learning_rate"],
        "momentum": FINAL_CONFIG["momentum"], "weight_decay": FINAL_CONFIG["weight_decay"],
        "patience": FINAL_CONFIG["patience"], "flipud": FINAL_CONFIG.get("flipud", 0.0),
    }
    result = run_training(
        model_arch=FINAL_CONFIG["model_arch"],
        data_yaml=PREPARED_DIR / "data.yaml",
        output_project=PROJECT_ROOT / "runs" / "detect" / "final",
        run_name="final_model_notebook_rerun",
        image_size=FINAL_CONFIG["image_size"],
        batch_size=FINAL_CONFIG["batch_size"],
        epochs=FINAL_CONFIG["epochs"],
        device=detect_device(),
        seed=SEED,
        workers=FINAL_CONFIG["workers"],
        fraction=FINAL_CONFIG["fraction"],
        plots=True,
        validate=True,
        extra_train_kwargs=extra_kwargs,
    )
    FINAL_WEIGHTS_PATH = Path(result["best_weights"])
    print(f"Pelatihan selesai. Weights terbaik: {FINAL_WEIGHTS_PATH}")

### Catatan pelatihan final

Rekaman resmi proses pelatihan disimpan pada
`artifacts/reports/block15_final_training_summary.json`.

In [ ]:
with open(REPORTS_DIR / "block15_final_training_summary.json") as f:
    training_summary = json.load(f)

print(f"Git commit saat pelatihan : {training_summary['git_commit']}")
print(f"Hash manifest dataset     : {training_summary['dataset_manifest_hash']}")
print(f"Perangkat                 : {training_summary['device']}")
print(f"Jumlah epoch              : {training_summary['config_used']['epochs']}")
print(f"Durasi                    : {training_summary['duration_seconds'] / 3600:.2f} jam")
print(f"Validasi muat proses bersih: {'LULUS' if training_summary['clean_process_load_validation']['success'] else 'GAGAL'}")

### Kurva pelatihan

Kurva berikut dibaca langsung dari `results.csv` yang dihasilkan proses
pelatihan, bukan dari angka yang diketik ulang.

In [ ]:
results_csv = PROJECT_ROOT / "runs" / "detect" / "final" / "final_model" / "results.csv"
history = {}
with open(results_csv) as f:
    for row in csv.DictReader(f):
        for k, v in row.items():
            history.setdefault(k.strip(), []).append(float(v))

fig_path = plot_training_curves(history, FIGURES_DIR / "training_curves.png")
plt.figure(figsize=(14, 5))
plt.imshow(mpimg.imread(fig_path))
plt.axis("off")
plt.show()

best_epoch = max(range(len(history["metrics/mAP50(B)"])), key=lambda i: history["metrics/mAP50(B)"][i])
print(f"mAP@0.5 tertinggi  : {history['metrics/mAP50(B)'][best_epoch]:.4f} pada epoch {int(history['epoch'][best_epoch])}")
print(f"mAP@0.5 epoch akhir: {history['metrics/mAP50(B)'][-1]:.4f}")

Komponen *loss* pada data latih menurun secara konsisten sepanjang 50
*epoch* tanpa lonjakan yang menandakan ketidakstabilan. Pada sisi validasi,
kurva mAP@0.5 meningkat tajam pada fase awal, lalu melandai pada sepertiga
terakhir pelatihan.

Pelandaian tersebut menunjukkan bahwa perolehan tambahan dari *epoch*
berikutnya semakin kecil pada konfigurasi ini. Tidak ditemukan pola
penurunan metrik validasi yang disertai penurunan *loss* latih secara
bersamaan, sehingga *overfitting* tidak diklaim berdasarkan data yang
tersedia.

## 14. Evaluasi

Evaluasi dijalankan melalui `scripts/evaluate.py` sebagai *subprocess*.
Pemisahan proses ini bukan sekadar preferensi gaya: uji reproduksi pada
lingkungan bersih menemukan bahwa menjalankan `val()` bawaan Ultralytics dan
pengumpulan prediksi untuk *F1* lokal di dalam satu proses yang sama merusak
kondisi internal *backend* MPS pada perangkat ini. Karena itu setiap tahap
dijalankan pada proses tersendiri.

Data uji tidak pernah digunakan untuk penyetelan apa pun. Evaluasi pada
notebook ini hanya menggunakan *split* validasi.

In [ ]:
subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "scripts" / "evaluate.py"),
     "--weights", str(FINAL_WEIGHTS_PATH),
     "--split", "valid",
     "--prepared-dir", str(PREPARED_DIR),
     "--conf-threshold", "0.25"],
    check=True, cwd=PROJECT_ROOT,
)

with open(REPORTS_DIR / "evaluation_valid.json") as f:
    eval_report = json.load(f)

### Hasil akhir

Dua metrik penilaian yang disebut regulasi adalah mAP@0.5 dan *F1-Score*.
Perlu dibedakan secara tegas:

- **mAP@0.5** dihitung menggunakan implementasi bawaan Ultralytics dan
  diperlakukan sebagai sumber kebenaran pada notebook ini.
- ***F1-score* lokal** merupakan implementasi internal project menggunakan
  pencocokan *greedy* dengan IoU minimal 0,5 pada *confidence threshold*
  tertentu. Regulasi yang tersedia tidak merinci *threshold* maupun mekanisme
  pencocokan yang dipakai panitia, sehingga nilai ini tidak boleh disebut
  sebagai skor resmi lomba.

In [ ]:
native = eval_report["native_metrics"]
local = eval_report["local_f1_metrics"]

print(f"{'Metrik':46s} {'Nilai':>12s}")
print(f"{'mAP@0.5':46s} {native['mAP50']:12.4f}")
print(f"{'mAP@0.5:0.95':46s} {native['mAP50_95']:12.4f}")
print(f"{'Precision (titik best-F1 internal Ultralytics)':46s} {native['precision_at_internal_best_f1_point']:12.4f}")
print(f"{'Recall (titik best-F1 internal Ultralytics)':46s} {native['recall_at_internal_best_f1_point']:12.4f}")
print(f"{'F1 lokal pada confidence 0,25':46s} {local['overall']['f1']:12.4f}")
print(f"{'  precision lokal':46s} {local['overall']['precision']:12.4f}")
print(f"{'  recall lokal':46s} {local['overall']['recall']:12.4f}")

model_size_mb = FINAL_WEIGHTS_PATH.stat().st_size / (1024 * 1024)
print(f"\n{'Ukuran model':46s} {model_size_mb:11.2f} MB")
print(f"{'Ukuran citra masukan':46s} {FINAL_CONFIG['image_size']:12d}")
print(f"{'Jumlah epoch':46s} {FINAL_CONFIG['epochs']:12d}")
print(f"{'Perangkat':46s} {training_summary['device']:>12s}")

**Catatan penting mengenai stabilitas metrik.** Pengulangan evaluasi pada
*checkpoint* yang sama menghasilkan nilai mAP@0.5 yang identik hingga digit
terakhir, yaitu 0,6276771766514752, pada empat kali pengulangan. Sebaliknya,
*F1* lokal pada *threshold* tetap bervariasi antar pengulangan dalam rentang
sekitar 0,22 sampai 0,42.

Pola ini konsisten dengan sifat nondeterministik *backend* MPS yang
terdokumentasi pada project ini, yang tampaknya juga memengaruhi tahap
*inference*, bukan hanya tahap pelatihan. Penjelasan tersebut belum
dibuktikan melalui eksperimen terkontrol, sehingga disajikan sebagai
indikasi. Untuk keperluan audit, mAP@0.5 merupakan titik perbandingan yang
stabil, sedangkan *F1* lokal sebaiknya dibaca sebagai metrik sekunder yang
disertai rentang.

## 15. Hasil Per Kelas

Angka agregat dapat menyembunyikan perbedaan besar antar kelas. Bagian ini
membongkar hasil tersebut.

In [ ]:
per_class_ap = native["per_class_AP50"]
fig_path = plot_per_class_ap(per_class_ap, FIGURES_DIR / "per_class_ap.png")
plt.figure(figsize=(10, 6))
plt.imshow(mpimg.imread(fig_path))
plt.axis("off")
plt.show()

print(f"{'Kelas':28s} {'AP@0.5':>8s} {'Instance train':>15s}")
for cls, ap in sorted(per_class_ap.items(), key=lambda kv: kv[1], reverse=True):
    print(f"{cls:28s} {ap:8.4f} {train_imb.per_class_instances[cls]:15d}")

Kelas dengan performa tertinggi adalah Narrow brown, False smut, Leaf
roller, dan Healthy, seluruhnya di atas AP@0.5 sebesar 0,87. Kelas dengan
performa terendah adalah Brown spot, Leaf scald, dan Bacterial leaf blight,
seluruhnya di bawah 0,39.

Yang menarik, Narrow brown justru merupakan kelas dengan jumlah *instance*
paling sedikit pada *split* latih, yaitu 222, tetapi memperoleh AP@0.5
tertinggi. Sebaliknya Brown spot memiliki jumlah *instance* terbanyak, yaitu
5.010, namun memperoleh AP@0.5 terendah. Temuan ini menunjukkan bahwa jumlah
data bukan penentu tunggal performa, dan hubungan tersebut diperiksa lebih
lanjut pada Bagian 17.

## 16. Analisis Kesalahan

Bagian ini menggunakan hasil `scripts/run_error_analysis.py`, yang
mengelompokkan kesalahan menjadi *true positive*, salah kelas,
*false positive* terhadap latar belakang, dan *false negative*.

In [ ]:
with open(REPORTS_DIR / "block13_error_analysis.json") as f:
    error_analysis = json.load(f)

counts = error_analysis["counts"]
for k, v in counts.items():
    print(f"{k:34s}: {v}")

small = error_analysis["small_object_miss_analysis"]
print(f"\nMedian luas bbox keseluruhan     : {small['overall_median_gt_area_px2']} px2")
print(f"Median luas bbox yang terlewat   : {small['false_negative_median_area_px2']} px2")

crowd = error_analysis["crowded_vs_sparse_scene_fn_rate"]
print(f"\nRasio false negative scene padat : {crowd['crowded_scene_fn_rate']}")
print(f"Rasio false negative scene jarang: {crowd['sparse_scene_fn_rate']}")

In [ ]:
fp_examples = sorted((PROJECT_ROOT / "artifacts" / "figures" / "error_analysis").glob("bg_fp_*.jpg"))[:2]
fn_examples = sorted((PROJECT_ROOT / "artifacts" / "figures" / "error_analysis").glob("fn_*.jpg"))[:2]
examples = [(p, "False positive latar belakang") for p in fp_examples]
examples += [(p, "False negative") for p in fn_examples]

if examples:
    fig, axes = plt.subplots(1, len(examples), figsize=(5 * len(examples), 5))
    if len(examples) == 1:
        axes = [axes]
    for ax, (path, label) in zip(axes, examples):
        ax.imshow(mpimg.imread(path))
        ax.set_title(label, fontsize=10)
        ax.axis("off")
    plt.suptitle("Contoh kesalahan model pada split validasi", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Belum ada figur analisis kesalahan. Jalankan scripts/run_error_analysis.py.")

Dua pola kesalahan menonjol dan keduanya didukung angka:

1. **Objek yang terlewat cenderung lebih kecil daripada rata-rata.** Median
   luas *bounding box* yang gagal terdeteksi jauh di bawah median luas
   seluruh *bounding box*. Temuan ini berasal dari kesalahan aktual, bukan
   sekadar dugaan dari angka agregat, dan konsisten dengan dominasi objek
   kecil yang ditemukan pada Bagian 6.5.
2. **Adegan padat memiliki rasio *false negative* lebih tinggi** daripada
   adegan jarang. Kondisi ini konsisten dengan poin pertama, karena objek
   kecil yang saling berdekatan merupakan kasus tersulit.

## 17. Hubungan Karakteristik Data dengan Performa

Bagian ini menguji secara eksplisit dugaan umum bahwa kelas dengan data
lebih banyak akan memperoleh performa lebih baik.

In [ ]:
fig_path = plot_instances_vs_performance(
    train_imb.per_class_instances, per_class_ap, FIGURES_DIR / "instances_vs_ap.png"
)
plt.figure(figsize=(10, 7))
plt.imshow(mpimg.imread(fig_path))
plt.axis("off")
plt.show()

import statistics as st

classes = [c for c in CANONICAL_CLASSES if c in per_class_ap]
xs = [train_imb.per_class_instances[c] for c in classes]
ys = [per_class_ap[c] for c in classes]
try:
    corr = st.correlation(xs, ys)
    print(f"Korelasi Pearson antara jumlah instance dan AP@0.5: {corr:.4f}")
except Exception as exc:
    print(f"Korelasi tidak dapat dihitung: {exc}")

Korelasi Pearson antara jumlah *instance* pada *split* latih dan AP@0.5 per
kelas bernilai sekitar -0,51, yaitu korelasi negatif dengan kekuatan sedang.
Arah hubungannya berlawanan dengan dugaan umum: pada dataset ini, kelas
dengan jumlah data lebih banyak justru cenderung memperoleh AP@0.5 lebih
rendah.

Beberapa peringatan penting sebelum menafsirkan angka tersebut. Pertama,
hubungan ini bersifat **observasional**, bukan kausal, dan tidak ada
eksperimen terkontrol yang dilakukan untuk mengujinya. Kedua, korelasi
dihitung hanya dari 11 titik data, sehingga sangat sensitif terhadap
beberapa kelas ekstrem dan tidak dapat dianggap sebagai bukti kuat. Ketiga,
arah negatif ini kemungkinan besar merupakan gejala dari variabel lain yang
kebetulan berkorelasi dengan jumlah data, bukan bukti bahwa menambah data
merugikan.

Yang dapat disimpulkan secara aman hanyalah bahwa jumlah data per kelas
tidak cukup untuk menjelaskan perbedaan performa antar kelas pada kasus
ini, sehingga faktor lain perlu dipertimbangkan.

Faktor lain yang secara masuk akal dapat berkontribusi, dan sebagiannya
didukung temuan pada bagian sebelumnya:

- **Ukuran objek.** Brown spot berupa bercak kecil yang tersebar, dan kelas
  ini memiliki rasio *instance* per citra tertinggi, yaitu sekitar 4,5
  *instance* per citra. Kombinasi objek kecil dan adegan padat merupakan
  kondisi yang terbukti paling sulit pada Bagian 16.
- **Kekhasan visual.** Narrow brown dan False smut memiliki penampakan yang
  relatif khas, sedangkan beberapa penyakit bercak daun memiliki kemiripan
  visual satu sama lain.
- **Konsistensi anotasi.** Objek kecil yang banyak pada satu citra lebih
  rentan terhadap variasi cara anotasi dilakukan.

## 18. Kelebihan dan Keterbatasan

### Kelebihan pendekatan

1. **Kepatuhan yang dapat diverifikasi pada tingkat kode.** Larangan
   *external pretrained weights* tidak hanya dinyatakan, tetapi ditegakkan
   oleh fungsi yang menolak berjalan bila dilanggar, ditambah `YOLO_OFFLINE`
   yang membuat setiap upaya pengunduhan gagal secara keras.
2. **Pemetaan canonical yang gagal secara keras.** Kategori mentah yang
   tidak dikenali menghentikan proses, sehingga perubahan dataset tidak
   lolos diam-diam.
3. **Penyiapan data yang deterministik dan tidak merusak.** Dataset mentah
   tidak pernah diubah, dan *manifest* dapat dihasilkan ulang secara identik.
4. **Keputusan konfigurasi berbasis eksperimen tercatat.** 21 percobaan
   terdokumentasi lengkap dengan *seed*, *hyperparameter*, dan *commit*.
5. **Jejak audit yang lengkap.** Setiap tahap menghasilkan laporan
   terstruktur yang dapat diperiksa ulang.
6. **Keterbatasan dilaporkan apa adanya**, termasuk ketidakstabilan *F1*
   lokal yang tidak menguntungkan bagi penyajian hasil.

### Keterbatasan

1. **Model dilatih dari nol.** Tanpa *pretrained weights*, model tidak
   mewarisi representasi visual umum. Ini merupakan konsekuensi aturan
   kompetisi, bukan pilihan desain.
2. **Nondeterminisme *backend* MPS.** Operasi `scatter_reduce_mps` dan
   `index_put_with_accumulate_mps` tidak memiliki implementasi deterministik
   pada perangkat ini, sehingga reproduksi bit per bit pada pelatihan tidak
   diklaim.
3. **Ketidakstabilan *F1* lokal.** Nilai bervariasi pada rentang 0,22 sampai
   0,42 antar pengulangan pada *checkpoint* yang sama, dan penyebab pastinya
   belum ditelusuri tuntas.
4. **Performa antar kelas tidak merata.** Selisih AP@0.5 antara kelas
   terbaik dan terburuk melebihi 0,67.
5. **Ekstrapolasi dari skala penyaringan.** Pemilihan *hyperparameter*
   divalidasi pada sebagian data dan sedikit *epoch*, lalu diterapkan pada
   skala penuh. Perilaku skala penuh hanya teramati langsung untuk
   konfigurasi final.
6. **Kandidat duplikat yang belum diverifikasi.** Kemiripan berbasis
   *perceptual hash* tidak diperiksa satu per satu secara visual.
7. **Belum ada validasi eksternal.** Model belum diuji pada data di luar
   dataset kompetisi, sehingga kemampuan generalisasi ke kondisi lapangan
   yang berbeda belum diketahui.
8. **Anggaran komputasi terbatas.** Seluruh pekerjaan dijalankan pada satu
   laptop, yang membatasi ukuran model dan jumlah percobaan.

## 19. Reproducibility dan Audit

Tiga tingkat reproduksi perlu dibedakan agar klaim tidak melampaui bukti:

1. **Prapemrosesan yang dapat direproduksi.** Terverifikasi byte per byte.
   Menjalankan ulang penyiapan dataset dengan *seed* yang sama menghasilkan
   *manifest* yang identik.
2. **Konfigurasi yang dapat direproduksi.** Terverifikasi. Setiap percobaan
   mencatat *seed*, *hyperparameter*, arsitektur, *hash manifest*, dan
   *commit* Git.
3. **Reproduksi pelatihan bit per bit.** **Tidak diklaim**, karena
   nondeterminisme *backend* MPS yang sudah dijelaskan.

Risiko dari tingkat ketiga dikurangi melalui *seed* tetap, konfigurasi beku,
*manifest* tetap, validasi pemuatan pada proses bersih, *checksum* model,
dan kode yang terversi.

In [ ]:
def sha256_of_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

with open(REPORTS_DIR / "final_model_metadata.json") as f:
    model_meta = json.load(f)

checksum_live = sha256_of_file(FINAL_WEIGHTS_PATH)

print("Tabel provenance")
print(f"{'Artefak':26s} {'Nilai'}")
print(f"{'Commit saat pelatihan':26s} {training_summary['git_commit']}")
print(f"{'Commit saat notebook ini':26s} {env['git_commit']}")
print(f"{'Hash manifest dataset':26s} {training_summary['dataset_manifest_hash']}")
print(f"{'Versi pemetaan kelas':26s} {model_meta['mapping_version']}")
print(f"{'SHA-256 weights (live)':26s} {checksum_live}")
print(f"{'SHA-256 weights (tercatat)':26s} {model_meta['weights_checksum_sha256']}")
print(f"{'Ukuran weights':26s} {FINAL_WEIGHTS_PATH.stat().st_size} byte")

assert checksum_live == model_meta["weights_checksum_sha256"], "Checksum weights tidak cocok dengan catatan."
print("\nChecksum cocok dengan metadata yang tercatat.")

Perbedaan antara *commit* saat pelatihan dan *commit* saat notebook
dieksekusi merupakan hal yang wajar, karena dokumentasi terus diperbarui
setelah pelatihan selesai. Yang perlu dipastikan adalah kode evaluasi tidak
berubah di antara keduanya, dan hal tersebut diverifikasi melalui
perbandingan `git diff` pada berkas evaluasi, metrik, dan pelatihan, yang
hasilnya kosong. *Checksum* model juga identik, sehingga metrik yang
dilaporkan memang berasal dari *weights* yang sama dengan hasil pelatihan.

### Uji inferensi mandiri

Pengujian berikut memuat *weights* dari awal, terlepas dari kondisi proses
pelatihan, lalu menjalankan inferensi pada lima citra validasi yang dipilih
secara deterministik menggunakan *seed* global. Sampel tidak disaring
berdasarkan keberhasilan deteksi, sehingga citra tanpa deteksi pun tetap
ditampilkan apa adanya.

In [ ]:
from ultralytics import YOLO

inference_model = YOLO(str(FINAL_WEIGHTS_PATH))

valid_images = sorted((PREPARED_DIR / "valid" / "images").iterdir())
set_global_seed(SEED)
demo_paths = random.sample(valid_images, 5)

for path in demo_paths:
    results = inference_model.predict(str(path), verbose=False)
    boxes = results[0].boxes
    print(f"{path.name[:52]:54s} {len(boxes)} deteksi")
    for box in boxes:
        cls_name = CANONICAL_CLASSES[int(box.cls.item())]
        print(f"    {cls_name:28s} confidence {float(box.conf.item()):.3f}")

## 20. Kesimpulan

**Apa yang dibangun.** Sebuah pipeline deteksi penyakit tanaman padi untuk
11 kelas canonical, dari audit dataset mentah sampai model terlatih beserta
jejak auditnya, menggunakan arsitektur YOLOv8n yang dilatih sepenuhnya dari
nol tanpa *external pretrained weights*.

**Seberapa baik performanya.** Pada *split* validasi, model mencapai mAP@0.5
sebesar 0,6277 dan mAP@0.5:0.95 sebesar 0,3905. Performa tidak merata antar
kelas, berkisar dari 0,9631 untuk Narrow brown sampai 0,2909 untuk Brown
spot. *F1* lokal pada *confidence* 0,25 berada pada rentang 0,22 sampai
0,42 antar pengulangan, dan karena itu diperlakukan sebagai metrik sekunder.

**Kekuatan utama.** Kepatuhan terhadap aturan ditegakkan pada tingkat kode,
bukan sekadar dinyatakan; seluruh keputusan konfigurasi dapat ditelusuri ke
percobaan yang tercatat; dan keterbatasan dilaporkan apa adanya, termasuk
yang tidak menguntungkan.

**Kelemahan utama.** Performa rendah pada kelas dengan objek kecil yang
padat, ketidakstabilan metrik *F1* lokal pada perangkat yang digunakan, dan
belum adanya validasi di luar dataset kompetisi.

**Apa yang menjelaskan keterbatasan performa.** Bukti yang terkumpul
mengarah pada kombinasi tiga hal: dominasi objek kecil pada dataset, adegan
padat yang memperburuk *false negative*, serta pelatihan dari nol dengan
model berkapasitas kecil pada anggaran komputasi satu laptop. Jumlah data
per kelas ternyata bukan faktor penjelas utama, karena korelasinya dengan
AP@0.5 justru lemah dan negatif.

**Seberapa reproducible.** Prapemrosesan terverifikasi identik byte per
byte, konfigurasi tercatat lengkap, dan pemuatan model diverifikasi pada
proses bersih. Reproduksi pelatihan bit per bit tidak diklaim karena
keterbatasan *backend* MPS.

**Arah perbaikan berikutnya.** Berdasarkan bukti yang ada, prioritas yang
paling beralasan adalah penanganan objek kecil secara khusus, misalnya
melalui resolusi masukan yang lebih tinggi atau strategi *tiling*, disertai
verifikasi konsistensi anotasi pada kelas dengan kepadatan objek tertinggi.

## 21. Daftar Pustaka

Bagian ini diisi pada tahap penyuntingan referensi. Sitasi hanya ditambahkan
untuk pernyataan yang benar-benar bersumber dari literatur, dan tidak
ditambahkan untuk pernyataan yang berasal dari aturan kompetisi maupun dari
hasil eksekusi pada project ini.

## 22. Lampiran Teknis

### Struktur repository

```
agridata/
  configs/          konfigurasi eksperimen dan konfigurasi final beku
  src/agridata/     paket inti: dataset, training, metrics, analysis, visualization
  scripts/          titik masuk CLI untuk setiap tahap pipeline
  notebooks/        notebook submission ini
  artifacts/        audit, laporan, figur, log eksperimen
  tests/            uji unit
  weights/          informasi rilis model
```

### Perintah reproduksi

```bash
python3.11 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt

python scripts/prepare_dataset.py --dataset-root "Telepati 8.0 Datasets" --output-dir data/prepared --seed 42
python scripts/profile_dataset.py --dataset-root "Telepati 8.0 Datasets"
python scripts/run_final_training.py --config configs/final_model_config.yaml
python scripts/evaluate.py --weights runs/detect/final/final_model/weights/best.pt --split valid --conf-threshold 0.25
python scripts/run_error_analysis.py --weights runs/detect/final/final_model/weights/best.pt --split valid
```

### Artefak audit utama

| Berkas | Isi |
|---|---|
| `artifacts/audit/dataset_audit_report.md` | Audit forensik dataset mentah |
| `artifacts/audit/canonical_mapping_report.md` | Validasi pemetaan 11 kelas |
| `artifacts/audit/reproducibility_checklist.md` | Daftar periksa reproducibility |
| `artifacts/audit/block16_clean_reproduction_test.md` | Uji reproduksi lingkungan bersih |
| `artifacts/audit/submission_state_audit.md` | Audit kondisi paket submission |
| `artifacts/reports/dataset_profile.json` | Profiling dataset lengkap |
| `artifacts/reports/evaluation_valid.json` | Hasil evaluasi split validasi |
| `artifacts/reports/block13_error_analysis.json` | Analisis kesalahan |
| `artifacts/experiments/experiment_log.json` | 21 percobaan terkontrol |